In [1]:
!python -m pip install -e ..

Obtaining file:///Users/dkoffical/Documents/GitHub/cs321m_project
  Installing build dependencies ... - \ | / - done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... - \ done
  Preparing editable metadata (pyproject.toml) ... - \ done
  Using cached torch-2.2.2-cp310-none-macosx_10_9_x86_64.whl (150.8 MB)
  Using cached scipy-1.15.3-cp310-cp310-macosx_10_13_x86_64.whl (38.7 MB)
  Using cached scikit_learn-1.7.2-cp310-cp310-macosx_10_9_x86_64.whl (9.3 MB)
  Using cached pandas-2.3.3-cp310-cp310-macosx_10_9_x86_64.whl (11.6 MB)
  Using cached pyarrow-24.0.0.tar.gz (1.2 MB)
  Installing build dependencies ... - \ | / - \ | / - \ | done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... - done
  Using cached pyro_ppl-1.9.1-py3-none-any.whl (755 kB)
  Using cached huggingface_hub-1.16.4-py3-none-any.whl (668 kB)
  Using cached seaborn-0.13.2-py3-none-any.w

In [2]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from torch_measure.datasets import load
from torch_measure.models import Rasch, TwoPL, ThreePL
from torch_measure.data import ResponseMatrix
from torch_measure.viz import plot_icc, plot_response_heatmap, plot_information
import torch_measure
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

import warnings
warnings.filterwarnings("ignore")

Using device: cpu


### Load a response matrix

Choose one matrix preset by changing `KFACTOR_MATRIX`. Rows are models/subjects, columns are items, and values are the observed score/correctness used by the factor model.


In [3]:
import pandas as pd
import torch
from torch_measure.data import ResponseMatrix

MATRIX_PRESETS = {
    "mmlu_solver": {
        "prefix": "mmlu_pro_solver",
        "dir": "../benchmarks/mmlu/response_matrices",
        "benchmark_id": "mmlu_pro_solver",
        "item_content_field": "source",
        "item_id_field": "pair_id",
        "value": "correct: 1.0=true, 0.0=false, NaN=unparsed/missing",
    },
    "mmlu_judging": {
        "prefix": "mmlu_pro_judging",
        "dir": "../benchmarks/mmlu/response_matrices",
        "benchmark_id": "mmlu_pro_judging",
        "item_content_field": "source",
        "item_id_field": "pair_id/order",
        "value": "judge score/correctness; NaN=missing",
    },
    "safety_all_attacks": {
        "prefix": "safety_all_attacks",
        "dir": "../benchmarks/safety/final_solver/response_matrices",
        "benchmark_id": "safety_all_attacks",
        "item_content_field": "source",
        "item_id_field": "attack_family:input_index",
        "value": "score: 1.0=harmful/successful, 0.0=not harmful/unsuccessful, NaN=missing",
    },
    "code_solver": {
        "prefix": "livecodebench",
        "dir": "../benchmarks/code/response_matrices",
        "benchmark_id": "livecodebench",
        "item_content_field": "difficulty",
        "item_id_field": "question_id",
        "value": "correct: 1.0=passes public tests, 0.0=fails public tests, NaN=missing",
    },
    "code_judge": {
        "prefix": "codejudgebench_pairwise",
        "dir": "../benchmarks/code/response_matrices",
        "benchmark_id": "codejudgebench_pairwise",
        "item_content_field": "difficulty",
        "item_id_field": "split:question_id:pair_index:ordering",
        "value": "correct: 1.0=selected correct solution, 0.0=selected incorrect solution, NaN=unparsed",
    },
    "harmmetric_eval": {
        "prefix": "harmmetric_eval",
        "dir": "../benchmarks/HarmMetric_Eval/response_matrices",
        "benchmark_id": "harmmetric_eval",
        "item_content_field": "source",
        "item_id_field": "prompt_id",
        "value": "overall_effectiveness_score: soft/fractional score in [0, 1]",
    },
    "kudge_challenge": {
        "prefix": "kudge_challenge_easy_hard",
        "dir": "../benchmarks/kudge/results/kudge_challenge_easy_hard/response_matrices",
        "benchmark_id": "kudge_challenge_easy_hard",
        "item_content_field": "prompt",
        "item_id_field": "id",
        "value": "correct: 1.0=true, 0.0=false",
        "enrich_kudge": True,
    },
    "kudge_judge": {
        "prefix": "kudge_judge_easy_hard",
        "dir": "../benchmarks/kudge/results/kudge_judge_easy_hard/response_matrices",
        "benchmark_id": "kudge_judge_easy_hard",
        "item_content_field": "prompt",
        "item_id_field": "id",
        "value": "correct: 1.0=true, 0.0=false",
        "enrich_kudge": True,
    },
}

KFACTOR_MATRIX = "mmlu_solver"  # temp 3-repeat sensitivity check
config = MATRIX_PRESETS[KFACTOR_MATRIX]
matrix_dir = config["dir"]
prefix = config["prefix"]

matrix_path = f"{matrix_dir}/{prefix}_response_matrix.csv"
subject_meta_path = f"{matrix_dir}/{prefix}_subject_metadata.csv"
item_meta_path = f"{matrix_dir}/{prefix}_item_metadata.csv"

print(f"Loading {KFACTOR_MATRIX}")
print(matrix_path)

df = pd.read_csv(matrix_path, index_col=0)
subject_meta = pd.read_csv(subject_meta_path)
item_meta = pd.read_csv(item_meta_path)

def enrich_kudge_item_metadata(item_meta):
    """Add prompt/source text from the original KUDGE dataset when available."""
    try:
        from datasets import load_dataset
    except ImportError:
        print("datasets is not installed; item metadata will only include response-matrix fields.")
        return item_meta

    try:
        ds = load_dataset("amphora/kudge-challenge", split="train")
    except Exception as exc:
        print(f"Could not load amphora/kudge-challenge; item metadata will only include response-matrix fields: {exc}")
        return item_meta

    source_rows = []
    for row in ds:
        if row.get("subset") not in {"Korean-Easy", "Korean-Hard"}:
            continue
        source_rows.append({
            "item_id": str(row.get("id")),
            "prompt": row.get("prompt"),
            "chosen": row.get("chosen"),
            "rejected": row.get("rejected"),
        })

    source_meta = pd.DataFrame(source_rows).drop_duplicates("item_id")
    enriched = item_meta.copy()
    enriched["item_id"] = enriched["item_id"].astype(str)
    enriched = enriched.merge(source_meta, on="item_id", how="left")
    print(f"Enriched item metadata with prompt text for {enriched['prompt'].notna().sum()}/{len(enriched)} items")
    return enriched

if config.get("enrich_kudge"):
    item_meta = enrich_kudge_item_metadata(item_meta)

item_content_field = config.get("item_content_field")
if item_content_field in item_meta.columns:
    item_contents = list(item_meta[item_content_field].fillna("").astype(str))
else:
    item_contents = list(item_meta.iloc[:, 0].astype(str))

rm = ResponseMatrix(
    data=torch.tensor(df.values, dtype=torch.float32),
    subject_ids=list(df.index.astype(str)),
    item_ids=list(df.columns.astype(str)),
    item_contents=item_contents,
    subject_metadata=subject_meta.to_dict("records"),
    info={
        "benchmark_id": config["benchmark_id"],
        "item_id_field": config["item_id_field"],
        "value": config["value"],
    },
)

print(f"{rm.n_subjects} models x {rm.n_items} items, density = {rm.density:.1%}")


Loading mmlu_solver
../benchmarks/mmlu/response_matrices/mmlu_pro_solver_response_matrix.csv
10 models x 154 items, density = 88.2%


### K-Factor Fit

Fit `LogisticFM` with `K=1` and `K=2` on the response matrix. Rows stay as models/subjects and columns stay as items. Here `Z` is item easiness, so item difficulty is `-Z`.


In [4]:
import json
from pathlib import Path

import pandas as pd
import torch
from tqdm.auto import tqdm

from torch_measure.models import LogisticFM, predict_dense
from torch_measure.models.rotation import varimax_rotation
from torch_measure.metrics.functional import compute_all

# Shared dense matrix + observed mask for all K-factor fits.
data = rm.data.to(device).float()
mask = ~torch.isnan(data) & (data != -1)

print(f"K-factor data: {data.shape[0]} models x {data.shape[1]} items, observed={mask.float().mean().item():.1%}")


K-factor data: 10 models x 154 items, observed=88.2%


In [5]:
def make_heldout_split(mask, test_frac=0.2, seed=123):
    """Split observed response entries into train/eval masks."""
    observed = mask.nonzero(as_tuple=False)
    gen = torch.Generator(device="cpu").manual_seed(seed)
    perm = torch.randperm(observed.shape[0], generator=gen)
    n_eval = max(1, int(round(test_frac * observed.shape[0])))
    eval_idx = observed[perm[:n_eval]]

    train_mask = mask.clone()
    eval_mask = torch.zeros_like(mask, dtype=torch.bool)
    train_mask[eval_idx[:, 0], eval_idx[:, 1]] = False
    eval_mask[eval_idx[:, 0], eval_idx[:, 1]] = True
    return train_mask, eval_mask


def fit_logistic_fm_k(data, train_mask, k, device="cpu", max_epochs=1000, lr=0.03, seed=123, eval_mask=None, verbose=True):
    torch.manual_seed(seed + k)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed + k)

    model = LogisticFM(
        n_subjects=data.shape[0],
        n_items=data.shape[1],
        n_factors=k,
        device=device,
    )

    history = model.fit(
        data,
        mask=train_mask,
        method="mle",
        max_epochs=max_epochs,
        lr=lr,
        verbose=verbose,
    )

    with torch.no_grad():
        probs = predict_dense(model).detach()

    train_metrics = compute_all(probs, data, mask=train_mask)
    eval_metrics = compute_all(probs, data, mask=eval_mask) if eval_mask is not None else None

    return {
        "model": model,
        "history": history,
        "metrics": eval_metrics if eval_metrics is not None else train_metrics,
        "train_metrics": train_metrics,
        "eval_metrics": eval_metrics,
    }


kfactor_specs = {
    1: {"max_epochs": 1000, "lr": 0.03},
    2: {"max_epochs": 1000, "lr": 0.02},
}

HELDOUT_REPEATS = 3
HELDOUT_TEST_FRAC = 0.2

heldout_rows = []
for k, spec in kfactor_specs.items():
    for repeat in range(HELDOUT_REPEATS):
        split_seed = 123 + 1000 * repeat + k
        train_mask, eval_mask = make_heldout_split(mask, test_frac=HELDOUT_TEST_FRAC, seed=split_seed)
        print(f"\nHeld-out fit LogisticFM K={k}, repeat={repeat + 1}/{HELDOUT_REPEATS}")
        result = fit_logistic_fm_k(
            data,
            train_mask,
            k=k,
            device=device,
            max_epochs=spec["max_epochs"],
            lr=spec["lr"],
            seed=split_seed,
            eval_mask=eval_mask,
        )
        metrics = result["eval_metrics"]
        heldout_log_likelihood = metrics.get("log_likelihood")
        heldout_rows.append({
            "k": k,
            "repeat": repeat,
            "loss": -heldout_log_likelihood if heldout_log_likelihood is not None else None,
            "auc": metrics.get("auc"),
            "log_likelihood": heldout_log_likelihood,
            "brier": metrics.get("brier"),
            "ece": metrics.get("ece"),
        })

heldout_eval_raw = pd.DataFrame(heldout_rows)
heldout_eval_summary = (
    heldout_eval_raw
    .groupby("k", as_index=False)
    .agg(
        loss=("loss", "mean"),
        loss_std=("loss", "std"),
        auc=("auc", "mean"),
        auc_std=("auc", "std"),
        log_likelihood=("log_likelihood", "mean"),
        log_likelihood_std=("log_likelihood", "std"),
        brier=("brier", "mean"),
        brier_std=("brier", "std"),
        ece=("ece", "mean"),
        ece_std=("ece", "std"),
    )
)

# Final full-data fits are used for item difficulty estimates and exported tables.
kfactor_results = {}
for k, spec in kfactor_specs.items():
    print(f"\nFinal full-data fit LogisticFM K={k}")
    kfactor_results[k] = fit_logistic_fm_k(
        data,
        mask,
        k=k,
        device=device,
        max_epochs=spec["max_epochs"],
        lr=spec["lr"],
        seed=123,
    )

final_fit_summary = pd.DataFrame([
    {
        "k": k,
        "final_train_loss": result["history"]["losses"][-1],
        "final_train_auc": result["train_metrics"].get("auc"),
        "final_train_log_likelihood": result["train_metrics"].get("log_likelihood"),
        "final_train_brier": result["train_metrics"].get("brier"),
        "final_train_ece": result["train_metrics"].get("ece"),
    }
    for k, result in kfactor_results.items()
])

kfactor_fit_summary = heldout_eval_summary.merge(final_fit_summary, on="k", how="left")

display(kfactor_fit_summary.round(4))



Held-out fit LogisticFM K=1, repeat=1/3


MLE fitting: 100%|██████████| 1000/1000 [00:02<00:00, 482.11it/s, loss=0.324602]



Held-out fit LogisticFM K=1, repeat=2/3


MLE fitting: 100%|██████████| 1000/1000 [00:01<00:00, 570.86it/s, loss=0.318054]



Held-out fit LogisticFM K=1, repeat=3/3


...


Final full-data fit LogisticFM K=1


MLE fitting: 100%|██████████| 1000/1000 [00:02<00:00, 479.33it/s, loss=0.358283]



Final full-data fit LogisticFM K=2


MLE fitting: 100%|██████████| 1000/1000 [00:02<00:00, 475.75it/s, loss=0.216045]


,k,loss,loss_std,auc,auc_std,log_likelihood,log_likelihood_std,brier,brier_std,ece,ece_std,final_train_loss,final_train_auc,final_train_log_likelihood,final_train_brier,final_train_ece
0,1,1.5946,0.2682,0.7130,0.0305,-1.5946,0.2682,0.2685,0.0116,0.2170,0.0246,0.3583,0.8841,-0.3583,0.1208,0.0361
1,2,2.8737,0.3466,0.6534,0.0591,-2.8737,0.3466,0.3289,0.0190,0.3224,0.0209,0.2160,0.9716,-0.2160,0.0706,0.0249


### Item Difficulty With Laplace Uncertainty

For `LogisticFM`, `Z` is item easiness and `difficulty = -Z`. The uncertainty below uses a diagonal/conditional Laplace approximation: item factors and model factors are held fixed, and item-difficulty uncertainty is estimated from Bernoulli curvature `sum p(1-p)` over observed model responses.


In [6]:
def build_item_difficulty_table(result, rm, item_meta=None):
    model = result["model"]

    difficulty = model.difficulty.detach().cpu()
    difficulty_centered = difficulty - difficulty.mean()
    easiness_z = model.Z.detach().cpu()
    loadings = model.loadings.detach().cpu()

    # Rotate loadings for interpretability. For K=1 this is the identity.
    rotated_loadings, rotation = varimax_rotation(loadings)
    rotated_abilities = model.U.detach().cpu() @ rotation

    table = pd.DataFrame({
        "item_id": rm.item_ids,
        "difficulty": difficulty.numpy(),
        "difficulty_centered": difficulty_centered.numpy(),
        "easiness_z": easiness_z.numpy(),
    })

    for j in range(rotated_loadings.shape[1]):
        table[f"loading_factor_{j + 1}"] = rotated_loadings[:, j].numpy()

    loading_cols = [c for c in table.columns if c.startswith("loading_factor_")]
    table["dominant_factor"] = (
        table[loading_cols]
        .abs()
        .idxmax(axis=1)
        .str.replace("loading_factor_", "factor_", regex=False)
    )

    if item_meta is not None:
        table = table.merge(
            item_meta.reset_index(drop=True),
            left_index=True,
            right_index=True,
            how="left",
            suffixes=("", "_meta"),
        )

    model_factors = pd.DataFrame({"model": rm.subject_ids})
    for j in range(rotated_abilities.shape[1]):
        model_factors[f"factor_{j + 1}"] = rotated_abilities[:, j].numpy()

    return table, model_factors


item_difficulty_tables = {}
model_factor_tables = {}
for k, result in kfactor_results.items():
    item_table, model_table = build_item_difficulty_table(result, rm, item_meta=item_meta)
    item_difficulty_tables[k] = item_table
    model_factor_tables[k] = model_table

print("K=1 hardest items")
display(item_difficulty_tables[1].sort_values("difficulty_centered", ascending=False).head(20).round(3))

print("K=2 hardest items")
display(item_difficulty_tables[2].sort_values("difficulty_centered", ascending=False).head(20).round(3))


K=1 hardest items


,item_id,difficulty,difficulty_centered,easiness_z,loading_factor_1,dominant_factor,item_id_meta,pair_id,source,split,gold_letter
99,7978,8.713,8.128,-8.713,-8.823,factor_1,7978,bd6f87f2-cb60-58b3-8b27-4eceab2bfc79,mmlu-pro-math,gpt,B
34,2934,8.517,7.933,-8.517,15.464,factor_1,2934,6c5f9b09-193f-5070-9dfd-2dee1f69a9a3,mmlu-pro-biology,gpt,F
112,9524,7.165,6.581,-7.165,2.334,factor_1,9524,34e8d16b-9824-5373-b735-d25a3df21044,mmlu-pro-physics,gpt,I
16,1330,6.544,5.959,-6.544,2.132,factor_1,1330,50e6565c-07f5-57d6-80d8-028498a1251b,mmlu-pro-law,gpt,H
81,6222,6.530,5.946,-6.530,2.099,factor_1,6222,1c76021c-cb8e-5477-8f7e-88855d6dd547,mmlu-pro-health,gpt,J
38,3250,5.949,5.364,-5.949,1.968,factor_1,3250,49c0f568-1ac2-53dc-be78-f3eea93820fd,mmlu-pro-biology,gpt,J
110,9158,5.915,5.330,-5.915,1.979,factor_1,9158,e3de7dfc-4b9e-5476-b7af-92d7d00bf2d3,mmlu-pro-physics,gpt,E
24,2256,5.860,5.276,-5.860,0.060,factor_1,2256,cc8d0d03-7d7e-5728-8bd6-fa03589c976d,mmlu-pro-psychology,gpt,J
65,5056,5.735,5.151,-5.735,1.876,factor_1,5056,6ed13f8d-9733-5839-a628-22c6ec6c9f27,mmlu-pro-history,gpt,A
59,4891,5.649,5.064,-5.649,9.282,factor_1,4891,d9cc990f-fb19-521e-a8fa-68f3325f8730,mmlu-pro-history,gpt,F


K=2 hardest items


,item_id,difficulty,difficulty_centered,easiness_z,loading_factor_1,loading_factor_2,dominant_factor,item_id_meta,pair_id,source,split,gold_letter
70,5145,6.745,7.152,-6.745,7.542,8.455,factor_2,5145,b1614f17-39fb-564a-84c7-bcd7a2054dc8,mmlu-pro-other,gpt,A
65,5056,6.050,6.457,-6.050,-1.558,0.605,factor_1,5056,6ed13f8d-9733-5839-a628-22c6ec6c9f27,mmlu-pro-history,gpt,A
153,12212,5.982,6.389,-5.982,1.915,0.328,factor_1,12212,c7bb7c2b-6fb2-5254-a773-9046ca946006,mmlu-pro-engineering,gpt,J
103,8312,5.801,6.208,-5.801,-0.254,1.290,factor_2,8312,405f3561-1bb7-56f5-88a2-c28cec94c5d4,mmlu-pro-math,gpt,B
96,7584,5.666,6.073,-5.666,0.499,0.552,factor_2,7584,766bdf22-3677-53bf-9c46-4fb984430682,mmlu-pro-economics,gpt,J
110,9158,5.367,5.774,-5.367,-1.971,0.101,factor_1,9158,e3de7dfc-4b9e-5476-b7af-92d7d00bf2d3,mmlu-pro-physics,gpt,E
86,6728,5.366,5.773,-5.366,-0.102,-0.078,factor_1,6728,0a347026-2ec5-5879-9635-e6fc818d021b,mmlu-pro-health,gpt,D
24,2256,4.945,5.352,-4.945,-0.103,-0.078,factor_1,2256,cc8d0d03-7d7e-5728-8bd6-fa03589c976d,mmlu-pro-psychology,gpt,J
16,1330,4.867,5.274,-4.867,-1.214,0.503,factor_1,1330,50e6565c-07f5-57d6-80d8-028498a1251b,mmlu-pro-law,gpt,H
95,7568,4.767,5.174,-4.767,-1.635,-1.526,factor_1,7568,a5c0fd0a-7440-5461-9a32-c6c5ff574ae0,mmlu-pro-economics,gpt,D


In [7]:
def item_difficulty_laplace_se(model, data, mask):
    """Conditional diagonal Laplace SE for LogisticFM item difficulty.

    This treats fitted model factors/loadings as fixed and estimates curvature
    only for each item intercept/difficulty. For difficulty = -Z, the sign
    does not change the curvature.
    """
    data = data.to(model.device).float()
    mask = mask.to(model.device)

    with torch.no_grad():
        probs = predict_dense(model).clamp(1e-7, 1 - 1e-7)
        info = torch.where(mask, probs * (1 - probs), torch.zeros_like(probs)).sum(dim=0)
        raw_se = 1 / torch.sqrt(info.clamp_min(1e-8))

        # Our main reported difficulty is mean-centered. Under a diagonal
        # approximation, propagate uncertainty through d_j - mean(d).
        n_items = raw_se.numel()
        raw_var = raw_se.pow(2)
        centered_var = ((1 - 1 / n_items) ** 2) * raw_var + (raw_var.sum() - raw_var) / (n_items ** 2)
        centered_se = torch.sqrt(centered_var.clamp_min(1e-12))

    return raw_se.detach().cpu(), centered_se.detach().cpu()


item_difficulty_with_uncertainty = {}
for k, result in kfactor_results.items():
    raw_se, centered_se = item_difficulty_laplace_se(result["model"], data, mask)

    table = item_difficulty_tables[k].copy()
    table["difficulty_laplace_se"] = raw_se.numpy()
    table["difficulty_centered_laplace_se"] = centered_se.numpy()
    table["difficulty_centered_laplace_lo"] = table["difficulty_centered"] - 1.96 * table["difficulty_centered_laplace_se"]
    table["difficulty_centered_laplace_hi"] = table["difficulty_centered"] + 1.96 * table["difficulty_centered_laplace_se"]
    item_difficulty_with_uncertainty[k] = table.sort_values("difficulty_centered", ascending=False).reset_index(drop=True)

print("K=1 item difficulties with Laplace uncertainty")
display(item_difficulty_with_uncertainty[1].round(3))

print("K=2 item difficulties with Laplace uncertainty")
display(item_difficulty_with_uncertainty[2].round(3))


K=1 item difficulties with Laplace uncertainty


,item_id,difficulty,difficulty_centered,easiness_z,loading_factor_1,dominant_factor,item_id_meta,pair_id,source,split,gold_letter,difficulty_laplace_se,difficulty_centered_laplace_se,difficulty_centered_laplace_lo,difficulty_centered_laplace_hi
0,7978,8.713,8.128,-8.713,-8.823,factor_1,7978,bd6f87f2-cb60-58b3-8b27-4eceab2bfc79,mmlu-pro-math,gpt,B,2.827,2.824,2.593000,13.664000
1,2934,8.517,7.933,-8.517,15.464,factor_1,2934,6c5f9b09-193f-5070-9dfd-2dee1f69a9a3,mmlu-pro-biology,gpt,F,3.860,3.846,0.394000,15.471000
2,9524,7.165,6.581,-7.165,2.334,factor_1,9524,34e8d16b-9824-5373-b735-d25a3df21044,mmlu-pro-physics,gpt,I,13.331,13.247,-19.384001,32.544998
3,1330,6.544,5.959,-6.544,2.132,factor_1,1330,50e6565c-07f5-57d6-80d8-028498a1251b,mmlu-pro-law,gpt,H,8.846,8.793,-11.276000,23.194000
4,6222,6.530,5.946,-6.530,2.099,factor_1,6222,1c76021c-cb8e-5477-8f7e-88855d6dd547,mmlu-pro-health,gpt,J,7.559,7.515,-8.784000,20.674999
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149,10745,-5.217,-5.802,5.217,0.558,factor_1,10745,3b317691-c56f-50df-92a6-8ad2f444dfe4,mmlu-pro-computer science,gpt,C,3.995,3.980,-13.603000,1.998000
150,11335,-5.839,-6.423,5.839,0.614,factor_1,11335,3e2b3d28-4ac1-5898-ad61-f5a58cee8698,mmlu-pro-engineering,gpt,C,5.474,5.446,-17.098000,4.251000
151,384,-7.026,-7.611,7.026,4.520,factor_1,384,6a01ee27-4246-5bb7-b1b7-bbea2dc28deb,mmlu-pro-business,gpt,A,1.442,1.463,-10.477000,-4.744000
152,2243,-9.889,-10.474,9.889,11.497,factor_1,2243,803e5d58-0c62-5946-9738-ceeea2e4aaa2,mmlu-pro-psychology,gpt,I,2.036,2.044,-14.481000,-6.467000


K=2 item difficulties with Laplace uncertainty


,item_id,difficulty,difficulty_centered,easiness_z,loading_factor_1,loading_factor_2,dominant_factor,item_id_meta,pair_id,source,split,gold_letter,difficulty_laplace_se,difficulty_centered_laplace_se,difficulty_centered_laplace_lo,difficulty_centered_laplace_hi
0,5145,6.745,7.152,-6.745,7.542,8.455,factor_2,5145,b1614f17-39fb-564a-84c7-bcd7a2054dc8,mmlu-pro-other,gpt,A,1.786,1.895,3.438,10.867
1,5056,6.050,6.457,-6.050,-1.558,0.605,factor_1,5056,6ed13f8d-9733-5839-a628-22c6ec6c9f27,mmlu-pro-history,gpt,A,8.646,8.616,-10.429,23.344
2,12212,5.982,6.389,-5.982,1.915,0.328,factor_1,12212,c7bb7c2b-6fb2-5254-a773-9046ca946006,mmlu-pro-engineering,gpt,J,1.232,1.394,3.657,9.121
3,8312,5.801,6.208,-5.801,-0.254,1.290,factor_2,8312,405f3561-1bb7-56f5-88a2-c28cec94c5d4,mmlu-pro-math,gpt,B,6.789,6.778,-7.076,19.493
4,7584,5.666,6.073,-5.666,0.499,0.552,factor_2,7584,766bdf22-3677-53bf-9c46-4fb984430682,mmlu-pro-economics,gpt,J,4.323,4.347,-2.447,14.592
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149,8315,-9.075,-8.668,9.075,-2.618,-3.163,factor_2,8315,000ad3d2-6b2a-5bee-baf2-fdf780b4e068,mmlu-pro-math,gpt,D,1.860,1.965,-12.519,-4.816
150,7623,-9.757,-9.350,9.757,-1.108,3.994,factor_2,7623,3c6f3a85-c058-543a-8226-852972410970,mmlu-pro-economics,gpt,D,3.864,3.896,-16.986,-1.714
151,6956,-10.816,-10.409,10.816,-9.308,6.164,factor_1,6956,ba87d469-ec2e-5a8f-95e8-b3065f20b584,mmlu-pro-economics,gpt,H,2.155,2.242,-14.803,-6.014
152,6311,-12.437,-12.030,12.437,-5.671,3.683,factor_1,6311,d61248f3-8e80-5097-8892-10c086a492d2,mmlu-pro-health,gpt,B,1.287,1.442,-14.857,-9.203


In [8]:
# Optional saves
result_name = KFACTOR_MATRIX
out_dir = Path("results") / result_name
out_dir.mkdir(parents=True, exist_ok=True)

def item_score_records(df, item_ids):
    """Return {item_id: {model: observed score}} with NaN converted to None."""
    score_map = {}
    for item_id in item_ids:
        values = df[str(item_id)] if str(item_id) in df.columns else df[item_id]
        score_map[str(item_id)] = {
            str(model): (None if pd.isna(value) else float(value))
            for model, value in values.items()
        }
    return score_map

scores_by_item = item_score_records(df, rm.item_ids)

kfactor_fit_summary.to_csv(out_dir / f"{result_name}_kfactor_fit_summary.csv", index=False)
item_difficulty_json = {
    "matrix": result_name,
    "benchmark_id": rm.info.get("benchmark_id"),
    "item_id_field": rm.info.get("item_id_field"),
    "value": rm.info.get("value"),
    "fits": {},
}
for k, table in item_difficulty_with_uncertainty.items():
    table.to_csv(out_dir / f"{result_name}_kfactor_k{k}_item_difficulties_with_laplace_uncertainty.csv", index=False)
    model_factor_tables[k].to_csv(out_dir / f"{result_name}_kfactor_k{k}_model_factors.csv", index=False)

    records = table.astype(object).where(pd.notna(table), None).to_dict("records")
    for record in records:
        record["scores"] = scores_by_item.get(str(record["item_id"]), {})
    item_difficulty_json["fits"][f"k{k}"] = records

with open(out_dir / f"{result_name}_kfactor_item_difficulties_with_laplace_uncertainty.json", "w") as f:
    json.dump(item_difficulty_json, f, indent=2)

print(f"Saved K-factor tables to {out_dir.resolve()}")


Saved K-factor tables to /Users/dkoffical/Documents/GitHub/cs321m_project/K-Factor/results/mmlu_solver


In [9]:
item_difficulty_json['fits']

{'k1': [{'item_id': '7978',
   'difficulty': 8.713128089904785,
   'difficulty_centered': 8.128339767456055,
   'easiness_z': -8.713128089904785,
   'loading_factor_1': -8.822929382324219,
   'dominant_factor': 'factor_1',
   'item_id_meta': 7978,
   'pair_id': 'bd6f87f2-cb60-58b3-8b27-4eceab2bfc79',
   'source': 'mmlu-pro-math',
   'split': 'gpt',
   'gold_letter': 'B',
   'difficulty_laplace_se': 2.8270719051361084,
   'difficulty_centered_laplace_se': 2.8242735862731934,
   'difficulty_centered_laplace_lo': 2.5927634239196777,
   'difficulty_centered_laplace_hi': 13.663915634155273,
   'scores': {'claude_haiku_4_5_20251001': 0.0,
    'claude_sonnet_4_6': 0.0,
    'mistralai_ministral_3_14b_instruct_2512_bf16': 0.0,
    'mistralai_ministral_3_3b_instruct_2512_bf16': 1.0,
    'mistralai_ministral_3_8b_instruct_2512_bf16': 1.0,
    'qwen_qwen3_5_0_8b': None,
    'qwen_qwen3_5_27b': None,
    'qwen_qwen3_5_2b': None,
    'qwen_qwen3_5_4b': None,
    'qwen_qwen3_5_9b': None}},
  {'item_i